In [3]:
# ==========================================
# 1. KHAI BÁO THƯ VIỆN & CẤU HÌNH THIẾT BỊ
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. TIỀN XỬ LÝ & TẢI DỮ LIỆU (CIFAR-10)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=32, shuffle=False, num_workers=2)

# 3. ĐỊNH NGHĨA MẠNG CNN TỰ LÀM (CUSTOM CNN)

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# 4. HÀM TẠO VÀ ĐIỀU CHỈNH MÔ HÌNH (MODEL BUILDER)

def get_model(model_name, use_pretrained):
    weights = models.ResNet18_Weights.DEFAULT if use_pretrained else None
    vgg_weights = models.VGG16_Weights.DEFAULT if use_pretrained else None

    if model_name == 'resnet':
        model = models.resnet18(weights=weights)
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, 10)
        
    elif model_name == 'vgg':
        model = models.vgg16(weights=vgg_weights)
        num_ftrs = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(num_ftrs, 10)
        
    elif model_name == 'custom':
        model = SimpleCNN()
        
    return model.to(device)

# ==========================================
# 5. HÀM HUẤN LUYỆN VÀ ĐÁNH GIÁ (TRAINING LOOP)
# ==========================================
def train_and_eval(model, epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        start_time = time.time()
        
        # --- BƯỚC HUẤN LUYỆN (TRAINING) ---
        for inputs, labels in trainloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        # --- BƯỚC ĐÁNH GIÁ (EVALUATION) ---
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                outputs = model(inputs)
                
                _, predicted = torch.max(outputs.data, 1)
                
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100.0 * correct / total
        print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(trainloader):.4f} | Accuracy: {accuracy:.2f}%")

# ==========================================
# 6. KHỐI THỰC THI CHÍNH (MAIN EXECUTION)
# ==========================================
EPOCHS = 5

print("1. Đang huấn luyện Mạng CNN tự xây dựng (Custom CNN)...")
custom_cnn = get_model('custom', use_pretrained=False)
train_and_eval(custom_cnn, epochs=EPOCHS)

print("\n2. Đang huấn luyện ResNet-18 (Từ đầu / Không có Transfer Learning)...")
resnet_scratch = get_model('resnet', use_pretrained=False)
train_and_eval(resnet_scratch, epochs=EPOCHS)

print("\n3. Đang huấn luyện ResNet-18 (Dùng Transfer Learning / Pre-trained ImageNet)...")
resnet_pretrained = get_model('resnet', use_pretrained=True)
train_and_eval(resnet_pretrained, epochs=EPOCHS)

print("\n4. Đang huấn luyện VGG-16 (Từ đầu / Không có Transfer Learning)...")
vgg_scratch = get_model('vgg', use_pretrained=False)
train_and_eval(vgg_scratch, epochs=EPOCHS)

print("\n5. Đang huấn luyện VGG-16 (Dùng Transfer Learning / Pre-trained ImageNet)...")
vgg_pretrained = get_model('vgg', use_pretrained=True)
train_and_eval(vgg_pretrained, epochs=EPOCHS)

1. Đang huấn luyện Mạng CNN tự xây dựng (Custom CNN)...
Epoch 1/5 | Loss: 1.5115 | Accuracy: 54.20%
Epoch 2/5 | Loss: 1.1619 | Accuracy: 58.16%
Epoch 3/5 | Loss: 0.9632 | Accuracy: 58.17%
Epoch 4/5 | Loss: 0.7912 | Accuracy: 57.34%
Epoch 5/5 | Loss: 0.6070 | Accuracy: 56.99%

2. Đang huấn luyện ResNet-18 (Từ đầu / Không có Transfer Learning)...
Epoch 1/5 | Loss: 1.4151 | Accuracy: 63.67%
Epoch 2/5 | Loss: 0.8590 | Accuracy: 72.35%
Epoch 3/5 | Loss: 0.6429 | Accuracy: 79.18%
Epoch 4/5 | Loss: 0.5004 | Accuracy: 80.77%
Epoch 5/5 | Loss: 0.4026 | Accuracy: 83.94%

3. Đang huấn luyện ResNet-18 (Dùng Transfer Learning / Pre-trained ImageNet)...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 144MB/s] 


Epoch 1/5 | Loss: 0.6839 | Accuracy: 82.18%
Epoch 2/5 | Loss: 0.3975 | Accuracy: 86.31%
Epoch 3/5 | Loss: 0.2949 | Accuracy: 87.56%
Epoch 4/5 | Loss: 0.2127 | Accuracy: 89.50%
Epoch 5/5 | Loss: 0.1599 | Accuracy: 88.75%

4. Đang huấn luyện VGG-16 (Từ đầu / Không có Transfer Learning)...
Epoch 1/5 | Loss: 2.4476 | Accuracy: 10.00%
Epoch 2/5 | Loss: 2.3029 | Accuracy: 10.00%
Epoch 3/5 | Loss: 2.3029 | Accuracy: 10.00%
Epoch 4/5 | Loss: 2.3028 | Accuracy: 10.00%
Epoch 5/5 | Loss: 2.3028 | Accuracy: 10.00%

5. Đang huấn luyện VGG-16 (Dùng Transfer Learning / Pre-trained ImageNet)...
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 210MB/s]  


Epoch 1/5 | Loss: 1.8413 | Accuracy: 40.74%
Epoch 2/5 | Loss: 1.4991 | Accuracy: 49.21%
Epoch 3/5 | Loss: 1.2954 | Accuracy: 56.22%
Epoch 4/5 | Loss: 1.2044 | Accuracy: 59.67%
Epoch 5/5 | Loss: 1.1384 | Accuracy: 61.23%
